# Neural Network Tuning with Keras Tuner

A feed-forward neural network for binary classification on the census-income dataset, where the architecture and training hyperparameters are searched automatically with **Keras Tuner**. This is the deep-learning counterpart to the classical-model comparison in the tabular-classification notebook, on the same data.

**Techniques:** preprocessing pipelines, feed-forward neural network, automated hyperparameter search (Keras Tuner RandomSearch over layer sizes, dropout and learning rate), dropout regularisation, early stopping.

### 1. Setup and imports

In [ ]:
# Install Keras Tuner
!pip install -q -U keras-tuner

# Import libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from kerastuner.tuners import RandomSearch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from pathlib import Path
import zipfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 2.5 MB/s eta 0:00:00


<ipython-input-2-51a27bca76ed>:10: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch


### 2. Load, clean and engineer features
Load the data, drop redundant columns, group occupations into broader categories and remove missing values.

In [ ]:
# Set Random Seed for Reproducibility
seed = 123
np.random.seed(seed)
tf.random.set_seed(seed)

# Extract ZIP
with zipfile.ZipFile('data1.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/data')

# Load CSV
csv_path = next(Path('/content/data').glob('*.csv'))
dataset = pd.read_csv(csv_path)

# Drop irrelevant columns
# As required: remove 'education' and 'native-country' from the dataset
columns_to_drop = ['education', 'native-country']
existing_columns = dataset.columns
columns_to_drop = [col for col in columns_to_drop if col in existing_columns]
dataset.drop(columns_to_drop, axis=1, inplace=True)

# Group occupations
occupation_categories = {
    'Tech': ['Tech-support', 'Craft-repair', 'Machine-op-inspct', 'Transport-moving'],
    'Sales': ['Sales', 'Adm-clerical'],
    'Exec': ['Exec-managerial', 'Prof-specialty'],
    'Service': ['Handlers-cleaners', 'Protective-serv', 'Priv-house-serv'],
    'Other': ['Other-service', 'Farming-fishing', 'Armed-Forces']
}


In [ ]:
# Function to map each occupation into its group
def categorize_occupation(occupation):
    for group, jobs in occupation_categories.items():
        if occupation in jobs:
            return group
    return 'Other'
# Apply the mapping if occupation column exists
if 'occupation' in dataset.columns:
    dataset['job_category'] = dataset['occupation'].apply(categorize_occupation)

# Clean missing and encode categorical columns
dataset.replace('?', np.nan, inplace=True)
dataset.dropna(inplace=True)


### 3. Encode and split
Encode categorical variables and the target, then separate features from the label.

In [ ]:
# Label encode original categorical columns
label_encode_cols = ['workclass', 'marital-status', 'job_category', 'relationship', 'race', 'sex']
encoder = LabelEncoder()
for col in label_encode_cols:
    if col in dataset.columns:
        dataset[col] = encoder.fit_transform(dataset[col])

# Encode target column to numeric
dataset['target'] = dataset['target'].str.strip()  # remove leading/trailing spaces
label_encoder = LabelEncoder()
dataset['target'] = label_encoder.fit_transform(dataset['target'])

# Now split
X = dataset.drop('target', axis=1)
y = dataset['target']

# Identify columns by type
# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()


### 4. Preprocess and prepare the arrays
Scale numerical features and one-hot-encode categorical ones inside a pipeline, split into train/validation, and cast to float32 for TensorFlow.

In [ ]:
# Preprocessing pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

# Split train/val (70/30)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=seed)

X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)

# Convert to dense if it's sparse
if hasattr(X_train, 'toarray'):
    X_train = X_train.toarray()
    X_val = X_val.toarray()

print(f"X_train type: {type(X_train)}, dtype: {X_train.dtype}")
print(f"X_val type: {type(X_val)}, dtype: {X_val.dtype}")
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)


X_train type: <class 'numpy.ndarray'>, dtype: float64
X_val type: <class 'numpy.ndarray'>, dtype: float64


### 5. Define the search space
The model has four hidden layers whose sizes, dropout rates and learning rate are all tunable. Keras Tuner runs a random search to find the best combination by validation accuracy.

In [ ]:
# Build Feedforward Neural Network with Hyperparameter Tuning
def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))
    # Add 4 hidden layers with tunable units and dropout
    for i in range(4):
        model.add(layers.Dense(units=hp.Int(f'units_{i}', 32, 256, step=32), activation='relu'))
        model.add(layers.Dropout(rate=hp.Float(f'dropout_{i}', 0.1, 0.5, step=0.1)))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Float("learning_rate", 1e-4, 1e-2, sampling="LOG")),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model



#Set Up Hyperparameter Tuner
tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='custom_ffnn_tune'
)


### 6. Run the search

In [ ]:
# Search for best hyperparameters
tuner.search(X_train, y_train,
             validation_data=(X_val, y_val),
             epochs=50,
             callbacks=[keras.callbacks.EarlyStopping(patience=3)])


Trial 10 Complete [00h 02m 01s]
val_accuracy: 0.8560241460800171

Best val_accuracy So Far: 0.8574572503566742
Total elapsed time: 00h 21m 04s


### 7. Evaluate the best model

In [ ]:
# Evaluate best model
best_model = tuner.get_best_models(num_models=1)[0]
val_loss, val_acc = best_model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc:.4f}")

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


306/306 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8605 - loss: 0.3125
Validation Accuracy: 0.8575


### Results
The tuned network reaches about 85.7% validation accuracy. Because the best score found during tuning matches the final evaluation, and dropout plus early stopping were used, the model generalises well without overfitting.